## My SQL database

## Step 1: 
- Connect Python to the local MySQL server and create the database behind_the_label. The connection address is read from the .env file (listed in .gitignore), so no password is stored in the notebook or on GitHub.

In [1]:
import os                                            # read environment variables
import pandas as pd                                  # tables + to_sql / read_sql for MySQL
from dotenv import load_dotenv                       # reads the .env file with the secrets
from sqlalchemy import create_engine, text          # engine = connection, text() = raw SQL

load_dotenv("../.env", override=True)                # notebook runs from notebook/, .env is one level up
connection_string = os.getenv("CON_STRING")          # full address incl. database name (never print it!)

# 1. Connect to the SERVER only: cut the database name off the end of the address
server_string = connection_string.rsplit("/", 1)[0]  # "mysql+pymysql://user:pw@host:3306/behind_the_label" -> "...:3306"
server_engine = create_engine(server_string)

# 2. Create the database once (does nothing if it already exists)
with server_engine.begin() as conn:                  # begin() saves (commits) the change at the end
    conn.execute(text("CREATE DATABASE IF NOT EXISTS behind_the_label"))

# 3. Now connect to the database itself, with the original full address
engine = create_engine(connection_string)

# 4. Checks
with engine.connect() as conn:
    print("MySQL version:", conn.execute(text("SELECT VERSION()")).scalar())          # server version
    print("Current database:", conn.execute(text("SELECT DATABASE()")).scalar())      # should be behind_the_label

MySQL version: 8.0.46
Current database: behind_the_label


**Findings**
- The connection to the local MySQL 8.0 server works and the database behind_the_label was created. The connection address stays in the .env file (listed in .gitignore), so no password is stored in the repository.

## Step 2a: database design 

The flat California table repeats the same company, product and chemical information thousands of times. I split it into tables where one row = one thing, linked by keys:

- companies: one row per company
- products: one row per product (→ company)
- product_categories: one row per product and category (a product can have several)
- chemicals: one row per chemical (key: CasId)
- reports: one row per reported chemical entry (→ product, → chemical), with CAS number, EU flags, dates
- eu_status: one row per line in an EU annex (automatic ID; (annex, ref_no) is an index)
- eu_cas: one row per CAS number and annex (link between chemicals and the EU rules)
- ingredients and ingredient_functions: one row per INCI name, and one per ingredient and function

## Step 2b: Design checks

In [2]:
# 1. Load the flagged California table (index-free, as saved on Day 2)
ca = pd.read_csv("../data/clean/california_eu.csv")

# 2. All column names: I use them to write the table definitions
print(ca.columns.tolist())

# 3. Is each candidate key really unique? (True = can be a primary key)
print("companies:", ca.groupby("CompanyId")["CompanyName"].nunique().max() == 1)      # one name per company id?
print("products :", ca.groupby("CDPHId")["ProductName"].nunique().max() == 1)         # one name per product id?
print("chemicals:", ca.groupby("ChemicalId")["ChemicalName"].nunique().max() == 1)    # one name per chemical id?
print("report keys duplicated:", ca.duplicated(subset=["CDPHId", "ChemicalId"]).sum())  # 0 = (product, chemical) is unique

# 4. Sizes of the future tables
print(ca["CompanyId"].nunique(), ca["CDPHId"].nunique(), ca["ChemicalId"].nunique(), len(ca))

['CDPHId', 'ProductName', 'CSFId', 'CSF', 'CompanyId', 'CompanyName', 'BrandName', 'PrimaryCategoryId', 'PrimaryCategory', 'SubCategoryId', 'SubCategory', 'CasId', 'CasNumber', 'ChemicalId', 'ChemicalName', 'InitialDateReported', 'MostRecentDateReported', 'DiscontinuedDate', 'ChemicalCreatedAt', 'ChemicalUpdatedAt', 'ChemicalDateRemoved', 'ChemicalCount', 'is_removed', 'is_discontinued', 'is_tio2', 'is_trade_secret', 'eu_banned', 'eu_restricted']
companies: True
products : True
chemicals: True
report keys duplicated: 55781
635 36972 58079 114381


In [3]:
# 1. Is CasId a proper chemical key? (one name per CasId, and how many distinct values)
print("CasId unique:", ca["CasId"].nunique(), "| names:", ca["ChemicalName"].nunique())
print("one name per CasId:", ca.groupby("CasId")["ChemicalName"].nunique().max() == 1)
print("one CasId per name:", ca.groupby("ChemicalName")["CasId"].nunique().max() == 1)

# 2. Which combination makes one report row unique? (0 = unique key)
print("product+CSF+chemical:", ca.duplicated(subset=["CDPHId", "CSFId", "ChemicalId"]).sum())
print("product+CSF+CasId   :", ca.duplicated(subset=["CDPHId", "CSFId", "CasId"]).sum())

# 3. Look at one repeated (product, chemical) pair to see what differs between the rows
dup = ca[ca.duplicated(subset=["CDPHId", "ChemicalId"], keep=False)].sort_values(["CDPHId", "ChemicalId"])
dup.head(6)[["CDPHId", "CSFId", "CSF", "ChemicalId", "CasId", "ChemicalName", "MostRecentDateReported"]]

CasId unique: 134 | names: 123
one name per CasId: True
one CasId per name: False
product+CSF+chemical: 9078
product+CSF+CasId   : 17588


,CDPHId,CSFId,CSF,ChemicalId,CasId,ChemicalName,MostRecentDateReported
6,7,310.0,5858-81-1,12178,656,Titanium dioxide,2010-12-30
7,7,311.0,D&C RED 7 CALCIUM LAKE,12178,656,Titanium dioxide,2010-12-30
8,7,312.0,D&C RED 28,12178,656,Titanium dioxide,2010-12-30
12,12,315.0,D&C RED 22,17917,656,Titanium dioxide,2010-12-30
13,12,316.0,RED IRON OXIDE,17917,656,Titanium dioxide,2010-12-30
14,13,317.0,ULTRAMARINE BLUE,17918,656,Titanium dioxide,2010-12-30


In [4]:
# 1. Which product columns vary within one product id? (number of products affected)
for col in ["ProductName", "BrandName", "CompanyId", "PrimaryCategoryId", "PrimaryCategory", "SubCategoryId", "SubCategory"]:
    n = (ca.groupby("CDPHId")[col].nunique(dropna=False) > 1).sum()     # >1 distinct value for the same product
    print(col, n)

# 2. CasIds that contain several CAS numbers
multi = ca.groupby("CasId")["CasNumber"].nunique()
multi = multi[multi > 1]                                                  # keep only CasIds with 2 or more CAS
print("CasIds with several CAS:", len(multi))
print(ca[ca["CasId"].isin(multi.index)].groupby(["CasId", "ChemicalName"])["CasNumber"].unique())

# 3. Does the EU flag differ inside one CasId? (0 = consistent)
print("eu_banned inconsistent CasIds:", (ca.groupby("CasId")["eu_banned"].nunique() > 1).sum())

ProductName 0
BrandName 0
CompanyId 0
PrimaryCategoryId 1176
PrimaryCategory 1176
SubCategoryId 2670
SubCategory 2670
CasIds with several CAS: 7
CasId  ChemicalName                                                                                       
566    o-Phenylenediamine and its salts                                                                                       [95-54-5, 106-50-3]
610    Selenium sulfide                                                                                                    [7488-56-4, 7446-34-6]
773    Retinol/retinyl esters, when in daily dosages in excess of 10,000 IU, or 3,000 retinol equivalents.    [79-81-2, nan, 11103-57-4, 68-26-8]
888    Coal Tar                                                                                                            [8007-45-2, 8002-45-2]
893    Vitamin A palmitate                                                                                                     [79-81-2, 79-91-2]
938    Retinyl pal

In [5]:
# 1. The whole schema as one text; each statement ends with ";"
schema = """
SET FOREIGN_KEY_CHECKS = 0;
DROP TABLE IF EXISTS reports, product_categories, products, chemicals, companies, eu_cas, eu_status, ingredient_functions, ingredients;
SET FOREIGN_KEY_CHECKS = 1;

-- one row per company
CREATE TABLE companies (
    company_id   INT PRIMARY KEY,
    company_name VARCHAR(255) NOT NULL
);

-- one row per product; links to its company
CREATE TABLE products (
    cdph_id      INT PRIMARY KEY,
    product_name VARCHAR(500),
    brand_name   VARCHAR(255),
    company_id   INT NOT NULL,
    FOREIGN KEY (company_id) REFERENCES companies(company_id)
);

-- one row per product and category (a product can have several)
CREATE TABLE product_categories (
    product_category_id INT AUTO_INCREMENT PRIMARY KEY,
    cdph_id             INT NOT NULL,
    primary_category_id INT,
    primary_category    VARCHAR(150),
    sub_category_id     INT,
    sub_category        VARCHAR(150),
    FOREIGN KEY (cdph_id) REFERENCES products(cdph_id)
);

-- one row per chemical (CasId); only name-level information
CREATE TABLE chemicals (
    cas_id          INT PRIMARY KEY,
    chemical_name   VARCHAR(255) NOT NULL,
    is_tio2         BOOLEAN,
    is_trade_secret BOOLEAN
);

-- one row per reported chemical entry; CAS number and EU status are stored per row
CREATE TABLE reports (
    report_id                 INT AUTO_INCREMENT PRIMARY KEY,
    cdph_id                   INT NOT NULL,
    cas_id                    INT NOT NULL,
    cas_number                VARCHAR(20),
    eu_banned                 BOOLEAN,
    eu_restricted             BOOLEAN,
    csf_id                    INT,
    csf                       VARCHAR(500),
    chemical_entry_id         INT,
    initial_date_reported     DATE,
    most_recent_date_reported DATE,
    discontinued_date         DATE,
    chemical_created_at       DATE,
    chemical_updated_at       DATE,
    chemical_date_removed     DATE,
    chemical_count            INT,
    is_removed                BOOLEAN,
    is_discontinued           BOOLEAN,
    FOREIGN KEY (cdph_id) REFERENCES products(cdph_id),
    FOREIGN KEY (cas_id)  REFERENCES chemicals(cas_id)
);

-- one row per line in an EU annex (entry numbers can repeat, so we use an automatic ID)
CREATE TABLE eu_status (
    eu_status_id  INT AUTO_INCREMENT PRIMARY KEY,
    annex         VARCHAR(3)  NOT NULL,
    ref_no        VARCHAR(20) NOT NULL,
    name          TEXT,
    inci_name     TEXT,
    inci_examples TEXT,
    cas           TEXT,
    ec            TEXT,
    product_type  TEXT,
    max_conc      TEXT,
    other_limits  TEXT,
    warnings      TEXT,
    cmr           TEXT,
    sccs_opinion  TEXT,
    last_update   DATE,
    INDEX idx_annex_ref (annex, ref_no)
);

-- one row per CAS number in an annex (the link between chemicals and the EU rules)
CREATE TABLE eu_cas (
    annex      VARCHAR(3)  NOT NULL,
    cas_single VARCHAR(20) NOT NULL,
    ref_no     VARCHAR(20) NOT NULL,
    PRIMARY KEY (annex, cas_single),
    INDEX idx_annex_ref (annex, ref_no)
);

-- one row per cosmetic ingredient (key = CosIng reference number)
CREATE TABLE ingredients (
    ingredient_id INT PRIMARY KEY,
    inci_name     TEXT NOT NULL,
    inn_name      VARCHAR(500),
    ph_eur_name   VARCHAR(500),
    cas           VARCHAR(500),
    ec            VARCHAR(500),
    description   TEXT,
    restriction   TEXT,
    last_update   DATE,
    INDEX idx_inci (inci_name(255))
);

-- one row per ingredient and function
CREATE TABLE ingredient_functions (
    ingredient_id INT NOT NULL,
    function_name VARCHAR(150) NOT NULL,
    PRIMARY KEY (ingredient_id, function_name),
    FOREIGN KEY (ingredient_id) REFERENCES ingredients(ingredient_id)
);
"""

# 2. Save the script in the repository (documents the design for recruiters)
with open("../sql/schema.sql", "w", encoding="utf-8") as f:
    f.write(schema)

# 3. Remove comment lines first (a ";" inside a comment would break the split), then run every statement
sql_only = "\n".join(line for line in schema.splitlines()
                     if not line.strip().startswith("--"))      # keep only real SQL lines

with engine.begin() as conn:                                     # begin() commits at the end
    for statement in sql_only.split(";"):                        # one statement per ";"
        if statement.strip():                                    # ignore the empty text after the last ";"
            conn.execute(text(statement))

# 4. Check: list the tables that now exist
with engine.connect() as conn:
    print([r[0] for r in conn.execute(text("SHOW TABLES")).fetchall()])

['chemicals', 'companies', 'eu_cas', 'eu_status', 'ingredient_functions', 'ingredients', 'product_categories', 'products', 'reports']


## Warning
Re-running Step 3 deletes all rows first and reloads every table.

## Step3: load the clean CSV files into the tables
- Load the clean CSV files into the tables. I split the flat California table into companies, products, chemicals and reports, and I check the keys on the way, so nothing is silently lost. if_exists="append" makes pandas add rows to my tables instead of replacing them, so the keys stay in place.

### California tables

In [6]:
# 0. Empty all tables first (child tables first), so the cell can be re-run
with engine.begin() as conn:
    for t in ["reports", "product_categories", "products", "companies", "chemicals",
              "eu_cas", "eu_status", "ingredient_functions", "ingredients"]:
        conn.execute(text(f"DELETE FROM {t}"))

# 1. Load the flagged California table; the 6 date columns are read as real dates
date_cols = ["InitialDateReported", "MostRecentDateReported", "DiscontinuedDate",
             "ChemicalCreatedAt", "ChemicalUpdatedAt", "ChemicalDateRemoved"]
ca = pd.read_csv("../data/clean/california_eu.csv", parse_dates=date_cols)

# 2. companies: one row per company
companies = (ca[["CompanyId", "CompanyName"]].drop_duplicates()
               .rename(columns={"CompanyId": "company_id", "CompanyName": "company_name"}))

# 3. products: one row per product (name, brand, company never vary)
products = (ca[["CDPHId", "ProductName", "BrandName", "CompanyId"]].drop_duplicates()
              .rename(columns={"CDPHId": "cdph_id", "ProductName": "product_name",
                               "BrandName": "brand_name", "CompanyId": "company_id"}))
print("products:", len(products), "| unique ids:", products["cdph_id"].nunique())      # both 36,972

# 4. product_categories: one row per product and category combination
product_categories = (ca[["CDPHId", "PrimaryCategoryId", "PrimaryCategory", "SubCategoryId", "SubCategory"]]
                        .drop_duplicates()
                        .rename(columns={"CDPHId": "cdph_id", "PrimaryCategoryId": "primary_category_id",
                                         "PrimaryCategory": "primary_category", "SubCategoryId": "sub_category_id",
                                         "SubCategory": "sub_category"}))
for c in ["primary_category_id", "sub_category_id"]:
    product_categories[c] = product_categories[c].astype("Int64")                       # whole numbers, missing allowed
print("product_categories:", len(product_categories))

# 5. chemicals: one row per CasId (name-level information only)
chemicals = (ca[["CasId", "ChemicalName", "is_tio2", "is_trade_secret"]].drop_duplicates("CasId")
               .rename(columns={"CasId": "cas_id", "ChemicalName": "chemical_name"}))
print("chemicals:", len(chemicals))                                                    # 134

# 6. reports: all 114,381 rows, now with CAS number and EU flags
reports = ca[["CDPHId", "CasId", "CasNumber", "eu_banned", "eu_restricted", "CSFId", "CSF", "ChemicalId"]
             + date_cols + ["ChemicalCount", "is_removed", "is_discontinued"]].rename(columns={
    "CDPHId": "cdph_id", "CasId": "cas_id", "CasNumber": "cas_number", "CSFId": "csf_id", "CSF": "csf",
    "ChemicalId": "chemical_entry_id", "InitialDateReported": "initial_date_reported",
    "MostRecentDateReported": "most_recent_date_reported", "DiscontinuedDate": "discontinued_date",
    "ChemicalCreatedAt": "chemical_created_at", "ChemicalUpdatedAt": "chemical_updated_at",
    "ChemicalDateRemoved": "chemical_date_removed", "ChemicalCount": "chemical_count"})
reports["csf_id"] = reports["csf_id"].astype("Int64")

# 7. Write to MySQL: parents first, then children
companies.to_sql("companies", engine, if_exists="append", index=False)
products.to_sql("products", engine, if_exists="append", index=False)
product_categories.to_sql("product_categories", engine, if_exists="append", index=False)
chemicals.to_sql("chemicals", engine, if_exists="append", index=False)
reports.to_sql("reports", engine, if_exists="append", index=False, chunksize=5000)
print("California tables loaded")

products: 36972 | unique ids: 36972
product_categories: 40096
chemicals: 134
California tables loaded


### EU and ingredient tables

In [7]:
# 1. eu_status: read everything as text (ref_no is text in the table), parse the date
eu_status = pd.read_csv("../data/clean/eu_status.csv", dtype=str)
eu_status = eu_status.rename(columns={"last_uptodate": "last_update"})                   # works for either column name
eu_status["last_update"] = pd.to_datetime(eu_status["last_update"], format="%d/%m/%Y", errors="coerce")
print("entry numbers used in 2 rows:", eu_status.duplicated(subset=["annex", "ref_no"]).sum())   # expect 11, allowed now

# 2. eu_cas: keep only rows that have a CAS, and only the 3 columns of the table
eu_cas = pd.read_csv("../data/clean/eu_cas.csv", dtype=str)
eu_cas = eu_cas.dropna(subset=["cas_single"])[["annex", "cas_single", "ref_no"]]

# 3. ingredients: the CosIng reference number becomes the key
ingredients = pd.read_csv("../data/clean/inventory_names.csv")
ingredients["last_update"] = pd.to_datetime(ingredients["last_update"], dayfirst=True, errors="coerce")
ingredients = ingredients.rename(columns={"ref_no": "ingredient_id"})
print("ingredient ids unique:", ingredients["ingredient_id"].is_unique)         # must be True

# 4. functions: look up the ingredient_id for each INCI name, then drop the long name
functions = pd.read_csv("../data/clean/inventory_functions.csv").rename(columns={"function": "function_name"})
functions["ingredient_id"] = functions["inci_name"].map(ingredients.set_index("inci_name")["ingredient_id"])
print("functions without ingredient:", functions["ingredient_id"].isna().sum())  # must be 0
functions = functions[["ingredient_id", "function_name"]].drop_duplicates()

# 5. Empty the four EU/ingredient tables first (children first), so the cell can be re-run
with engine.begin() as conn:
    for t in ["eu_cas", "eu_status", "ingredient_functions", "ingredients"]:
        conn.execute(text(f"DELETE FROM {t}"))

# 6. Load one table at a time; on an error print only a short message and stop
for name, df in [("eu_status", eu_status), ("eu_cas", eu_cas),
                 ("ingredients", ingredients), ("ingredient_functions", functions)]:
    try:
        df.to_sql(name, engine, if_exists="append", index=False, chunksize=5000)
        print(name, "loaded:", len(df), "rows")
    except Exception as e:
        cause = e.__cause__ or e                          # pandas wraps the SQLAlchemy error: look one level deeper
        print(name, "FAILED:", str(getattr(cause, "orig", cause))[:300])   # the real MySQL message, cut to 300 characters
        break

# 7. Check: row count of every table (compare with the expected numbers)
with engine.connect() as conn:
    for t in ["companies", "products", "chemicals", "reports", "eu_status", "eu_cas", "ingredients", "ingredient_functions", "product_categories"]:
        print(t, conn.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar())

entry numbers used in 2 rows: 11
ingredient ids unique: True
functions without ingredient: 0
eu_status loaded: 2361 rows
eu_cas loaded: 2983 rows
ingredients loaded: 30079 rows
ingredient_functions loaded: 53278 rows
companies 635
products 36972
chemicals 134
reports 114381
eu_status 2361
eu_cas 2983
ingredients 30079
ingredient_functions 53278
product_categories 40096


**Findings:**
- In the EU annexes the entry number (annex + ref_no) is not unique: 11 numbers appear in two rows. eu_status therefore uses an automatic ID as primary key, and (annex, ref_no) is an index. eu_cas references the entry by annex and ref_no without a foreign key.
- Some INCI names are longer than 500 characters, so the name is not suitable as a primary key. The CosIng reference number becomes ingredient_id; the INCI name is indexed (first 255 characters) for matching with Sephora.

## Step 4: indexes and the function eu_status_of(cas)
- Indexes make joins on CAS number fast (without them MySQL scans every row). The function eu_status_of(cas) answers “what is the EU status of this CAS number?” with one priority rule: banned (Annex II) beats restricted (Annex III) beats listed (Annex IV-VI: colourants, preservatives, UV filters). It’s stored in the database, so Python, Tableau and Power BI can all use it.

In [8]:
# 1. Indexes on the columns we join and filter by (run this part only ONCE; a second run says "duplicate key name")
index_sql = """
CREATE INDEX idx_reports_cas     ON reports (cas_number);
CREATE INDEX idx_eu_cas_cas      ON eu_cas (cas_single);
CREATE INDEX idx_chem_name       ON chemicals (chemical_name);
CREATE INDEX idx_reports_removed ON reports (is_removed);
"""

# 2. The stored function (one statement, so we do not split it at ";")
function_sql = """
CREATE FUNCTION eu_status_of(p_cas VARCHAR(20)) RETURNS VARCHAR(20)
READS SQL DATA
BEGIN
    DECLARE best INT;
    -- smallest rank wins: 1 = Annex II, 2 = Annex III, 3 = Annex IV-VI
    SELECT MIN(CASE annex WHEN 'II' THEN 1 WHEN 'III' THEN 2 ELSE 3 END)
      INTO best FROM eu_cas WHERE cas_single = p_cas;
    RETURN CASE best WHEN 1 THEN 'banned'
                     WHEN 2 THEN 'restricted'
                     WHEN 3 THEN 'listed (IV-VI)'
                     ELSE 'not listed' END;
END
"""

# 3. Save both scripts in the repository (documents the database for recruiters)
with open("../sql/indexes_and_function.sql", "w", encoding="utf-8") as f:
    f.write(index_sql + "\nDROP FUNCTION IF EXISTS eu_status_of;\n" + function_sql + ";\n")

# 4. Run them
# 4a. Indexes: one statement per ";", each in its own transaction, so one "index exists" error does not stop the rest
for stmt in index_sql.split(";"):
    if stmt.strip():
        try:
            with engine.begin() as conn:
                conn.execute(text(stmt))
        except Exception:
            print("index exists already:", stmt.split()[2])     # a second run no longer fails

# 4b. The function: drop the old one first, then create it as ONE statement
with engine.begin() as conn:
    conn.execute(text("DROP FUNCTION IF EXISTS eu_status_of"))   # makes the function re-runnable
    conn.execute(text(function_sql))
# 5. Test the function with three CAS numbers: formaldehyde (banned), titanium dioxide, a made-up one
print(pd.read_sql(text("""SELECT eu_status_of('50-00-0')    AS formaldehyde,
                                 eu_status_of('13463-67-7') AS titanium_dioxide,
                                 eu_status_of('00-00-0')    AS made_up"""), engine))

# 6. EU status of all 114,381 California reports in one query
print(pd.read_sql(text("""SELECT eu_status_of(cas_number) AS eu_status, COUNT(*) AS reports
                          FROM reports GROUP BY eu_status ORDER BY reports DESC"""), engine))

  formaldehyde titanium_dioxide     made_up
0       banned       restricted  not listed
        eu_status  reports
0      restricted    97677
1      not listed    13595
2  listed (IV-VI)     2264
3          banned      845


**Findings:**
- The stored function works: of 114,381 reports, 845 contain an EU-banned chemical (matches the Python result), 97,677 a restricted one (mostly titanium dioxide, restricted for sprays and powders), 2,264 a chemical on a positive list (IV-VI), and 13,595 are not in any annex or have no CAS.

## Step 5: test queries (joins and GROUP BY)
- Four test queries show that the tables connect correctly and give first answers: which companies, which chemicals, which categories, and how removal differs by EU status. Counting DISTINCT products avoids double counting, because a product has many report rows.

In [9]:
# small helper: run a SQL text and return a table
def q(sql):
    return pd.read_sql(text(sql), engine)

# 1. Top 10 companies by number of products with an EU-banned chemical (joins reports -> products -> companies)
display(q("""
    SELECT c.company_name, COUNT(DISTINCT r.cdph_id) AS products_with_banned
    FROM reports r
    JOIN products  p ON p.cdph_id    = r.cdph_id        -- report -> product
    JOIN companies c ON c.company_id = p.company_id     -- product -> company
    WHERE r.eu_banned = 1
    GROUP BY c.company_name
    ORDER BY products_with_banned DESC
    LIMIT 10"""))

# 2. The 10 banned chemicals that occur in the most products
display(q("""
    SELECT ch.chemical_name, COUNT(DISTINCT r.cdph_id) AS products
    FROM reports r
    JOIN chemicals ch ON ch.cas_id = r.cas_id
    WHERE r.eu_banned = 1
    GROUP BY ch.chemical_name
    ORDER BY products DESC
    LIMIT 10"""))

# 3. Removal rate by EU status, without titanium dioxide (which is 82% of all rows)
display(q("""
    SELECT eu_status_of(r.cas_number) AS eu_status,
           COUNT(*)                   AS reports,
           ROUND(100 * AVG(r.is_removed), 1) AS removed_percent   -- AVG of 0/1 = share removed
    FROM reports r
    JOIN chemicals ch ON ch.cas_id = r.cas_id
    WHERE ch.is_tio2 = 0
    GROUP BY eu_status
    ORDER BY removed_percent DESC"""))

# 4. Product categories with the most banned-chemical products (a product with 2 categories counts in both)
display(q("""
    SELECT pc.primary_category, COUNT(DISTINCT b.cdph_id) AS products_with_banned
    FROM (SELECT DISTINCT cdph_id FROM reports WHERE eu_banned = 1) b    -- each affected product once
    JOIN product_categories pc ON pc.cdph_id = b.cdph_id
    GROUP BY pc.primary_category
    ORDER BY products_with_banned DESC
    LIMIT 10"""))

,company_name,products_with_banned
0,"La Prairie, Inc.",64
1,American International Industries,49
2,"Elizabeth Arden, Inc.",38
3,WORLD CLUB SUPPPLY CORP.,26
4,John Paul Mitchell Systems,18
5,Georgia-Pacific Consumer Products LP,15
6,Regis Corporation,15
7,"Betco Corporation, Ltd",11
8,Harmon Stores Inc.,10
9,Fisk Industries Inc,9


,chemical_name,products
0,Formaldehyde (gas),104
1,Benzophenone,95
2,"1,4-Dioxane",59
3,Styrene,52
4,Safrole,33
5,Ethylene oxide,31
6,Acetaldehyde,30
7,Diethanolamine,30
8,"N,N-Dimethyl-p-toluidine",12
9,Coal Tar,12


,eu_status,reports,removed_percent
0,banned,845,16.7
1,not listed,13241,7.2
2,listed (IV-VI),2264,4.5
3,restricted,4468,3.0


,primary_category,products_with_banned
0,Skin Care Products,119
1,Hair Care Products (non-coloring),97
2,Makeup Products (non-permanent),66
3,Nail Products,54
4,Bath Products,43
5,Fragrances,41
6,Personal Care Products,29
7,Sun-Related Products,10
8,Baby Products,6
9,Hair Coloring Products,6


**Findings**

- Removal rate by EU status (without TiO2): banned 16.7%, not listed 7.2%, positive lists 4.5%, restricted 3.0%. The stricter the EU rule, the more often the chemical was removed. This is a pattern only; the chi-square test (H1) follows.
- Most frequent banned chemicals: formaldehyde (104 products), benzophenone (95), 1,4-dioxane (59), styrene (52). Several are impurities or by-products.
 - Company and category counts are absolute, so they reflect the size of a company or category and not product quality. Skin care and hair care are the most affected categories.